# 03 · Reversible transformer at its maximum batch size

1. **Memory vs depth**: peak memory at a small fixed batch for L = 4, 8, 16, 32, Euler vs the chosen reversible variant. This checks the lesson's claim that depth drops out of the activation memory.
2. **Max batch**: search for the largest batch that fits and train on the same 50M tokens.

The token budget is fixed, so a larger batch means **fewer optimizer steps**. At roughly 1000 sequences per batch, 50M tokens is only about 100 steps. The LR is scaled by √(B/B_fixed) (capped at 3e-3), with 10% warm-up. Expect the final loss to be worse than run 2's: that is the trade-off being measured. Notebook 04 plots loss against both tokens and optimizer steps.

In [1]:
# --- Colab setup: GPU runtime (Runtime > Change runtime type > T4 GPU) ---
REPO_URL = "https://github.com/gaurkhare/gaurav-eagv5-s13.git"
import os, sys, json
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/era_a13"          # data + results persist across notebooks
    if not os.path.exists("/content/repo"):
        !git clone -q {REPO_URL} /content/repo
    os.chdir("/content/repo")
else:
    WORK = os.path.abspath("..")                      # running locally from notebooks/
    os.chdir(WORK)
sys.path.insert(0, os.getcwd())
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
DATA_DIR, RESULTS = f"{WORK}/data", f"{WORK}/results"
os.makedirs(RESULTS, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv 2>/dev/null || echo "no nvidia GPU"

Mounted at /content/drive
name, memory.total [MiB]
Tesla T4, 15360 MiB


In [2]:
import math, torch
from revllm import ModelConfig, TrainConfig, train, find_max_batch, fits
B_FIXED = json.load(open(f"{RESULTS}/batch.json"))["fixed_batch"]
vj = json.load(open(f"{RESULTS}/variant.json"))
BEST, KW = vj["best_variant"], vj["model_kwargs"]
print("variant:", BEST, KW, "| fixed batch:", B_FIXED)

variant: leapfrog_h1 {'residual': 'leapfrog', 'h': 1.0} | fixed batch: 64


## Peak memory vs depth

In [3]:
B_DEPTH = 16
depth = {"euler": {}, BEST: {}}
for L in [4, 8, 16, 32]:
    for name, kw in [("euler", dict(residual="euler")), (BEST, KW)]:
        ok, peak = fits(ModelConfig(n_layers=L, **kw), B_DEPTH)
        depth[name][L] = peak
        print(f"L={L:2d} {name:22s} {'%.2f GB' % peak if ok else 'OOM'}")
json.dump({"batch": B_DEPTH, **depth}, open(f"{RESULTS}/mem_vs_depth.json", "w"), indent=1)

L= 4 euler                  2.50 GB
L= 4 leapfrog_h1            1.73 GB
L= 8 euler                  3.35 GB
L= 8 leapfrog_h1            1.78 GB
L=16 euler                  5.03 GB
L=16 leapfrog_h1            1.89 GB
L=32 euler                  8.40 GB
L=32 leapfrog_h1            2.11 GB


## Max batch search and training

In [4]:
mcfg = ModelConfig(**KW)
bmax, trials = find_max_batch(mcfg, start=B_FIXED)
print(f"reversible ({BEST}) max batch = {bmax}  ({bmax/B_FIXED:.1f}x the fixed batch)")
bj = json.load(open(f"{RESULTS}/batch.json")); bj.update(rev_max_batch=bmax, rev_trials=trials)
json.dump(bj, open(f"{RESULTS}/batch.json", "w"), indent=1)

  B=  128: ok  peak 4.18 GB
  B=  256: ok  peak 8.05 GB
  B=  512: OOM
  B=  384: ok  peak 11.92 GB
  B=  448: ok  peak 13.85 GB
  B=  480: OOM
  B=  464: OOM
  B=  456: ok  peak 14.10 GB
reversible (leapfrog_h1) max batch = 456  (7.1x the fixed batch)


In [5]:
def run(B):
    lr = min(3e-3, 1e-3 * math.sqrt(B / B_FIXED))
    steps = 50_000_000 // (B * mcfg.seq_len)
    print(f"B={B}, lr={lr:.2e}, optimizer steps={steps} (run 2 had {50_000_000 // (B_FIXED * mcfg.seq_len)})")
    cfg = TrainConfig(name="reversible_maxbatch", data_dir=DATA_DIR, out_dir=RESULTS, batch_size=B,
                      total_tokens=50_000_000, lr=lr, warmup_frac=0.1, eval_every=max(5, steps // 20),
                      log_every=max(1, min(20, steps // 100)), model=mcfg)
    return train(cfg)

# find_max_batch ran two full train steps at bmax, so the run should fit; the fallback only guards against
# allocator fragmentation late in a long run. Whatever batch actually trains is recorded next to bmax.
B_MAX, oom = bmax, False
try:
    res, model = run(B_MAX)
except torch.OutOfMemoryError:
    oom = True
if oom:
    # retry only after leaving the except block: until then the traceback keeps the failed run's
    # model, optimizer state and activations alive, and empty_cache() could not release them
    import gc; gc.collect(); torch.cuda.empty_cache()
    B_MAX = int(0.9 * bmax) // 8 * 8
    print(f"OOM at the searched max {bmax}; retrying at {B_MAX}")
    res, model = run(B_MAX)
bj = json.load(open(f"{RESULTS}/batch.json")); bj.update(rev_train_batch=B_MAX)
json.dump(bj, open(f"{RESULTS}/batch.json", "w"), indent=1)

B=456, lr=2.67e-03, optimizer steps=214 (run 2 had 1525)
[reversible_maxbatch] device=cuda amp=torch.bfloat16 params=20.16M residual=leapfrog (reversible backprop) B=456 T=512 steps=214 tokens=50.0M lr=0.00266927
OOM at the searched max 456; retrying at 408
B=408, lr=2.52e-03, optimizer steps=239 (run 2 had 1525)
[reversible_maxbatch] device=cuda amp=torch.bfloat16 params=20.16M residual=leapfrog (reversible backprop) B=408 T=512 steps=239 tokens=49.9M lr=0.00252488
step    10/239 loss 8.5376 (ema 9.8143) lr 1.10e-03 gnorm 1.94 9.4k tok/s peak 12.64 GB
  eval @ 11: val_loss 8.1447
step    20/239 loss 7.3008 (ema 8.8873) lr 2.20e-03 gnorm 0.74 9.4k tok/s peak 12.64 GB
  eval @ 22: val_loss 7.3658
step    30/239 loss 7.1649 (ema 8.2401) lr 2.52e-03 gnorm 0.69 9.4k tok/s peak 12.64 GB
  eval @ 33: val_loss 6.9883
step    40/239 loss 6.9032 (ema 7.7139) lr 2.49e-03 gnorm 0.60 9.3k tok/s peak 12.64 GB
  eval @ 44: val_loss 6.8262
step    50/239 loss 6.7029 (ema 7.3267) lr 2.44e-03 gnorm 0.5